In [44]:
import pandas as pd
import glob
import os
import lz4.frame

data_dir = './hl_btc_data/'
output_file = 'btc_5min_candles_final.csv'

all_data = []

for file_path in sorted(glob.glob(os.path.join(data_dir, "*.csv.lz4"))):
    try:
        with lz4.frame.open(file_path, 'rb') as f:
            # 0=Time, 5=day_ntl_vlm (USD), 9=mid_px
            df = pd.read_csv(f, header=None, on_bad_lines='skip')
            df_btc = df[df[1] == 'BTC'].copy()
            
            if not df_btc.empty:
                # Map exactly to your observed columns
                df_btc = df_btc.rename(columns={0: 'ts', 5: 'cum_vlm', 9: 'px'})
                df_btc['dt'] = pd.to_datetime(df_btc['ts'], unit='ms')
                df_btc = df_btc.set_index('dt')
                df_btc['px'] = pd.to_numeric(df_btc['px'], errors='coerce')
                df_btc['cum_vlm'] = pd.to_numeric(df_btc['cum_vlm'], errors='coerce')
                all_data.append(df_btc[['px', 'cum_vlm']])
    except:
        continue

if all_data:
    # 1. Combine and sort
    full_df = pd.concat(all_data).sort_index()
    
    # 2. Resample to 5-minute intervals (label='left' matches API "Open Time")
    resampled = full_df.resample('5min', closed='left', label='left').agg({
        'px': ['first', 'max', 'min', 'last'],
        'cum_vlm': 'last'
    })
    resampled.columns = ['open', 'high', 'low', 'close', 'cum_vlm']

    # 3. Calculate 5-minute USD Volume
    resampled['usd_vol'] = resampled['cum_vlm'].diff()
    # Handle Day Resets
    resampled.loc[resampled['usd_vol'] < 0, 'usd_vol'] = resampled['cum_vlm']
    resampled['usd_vol'] = resampled['usd_vol'].fillna(0)

    # 4. Convert USD to BTC Volume (Divide Notional by Close Price)
    resampled['volume'] = resampled['usd_vol'] / resampled['close']

    # 5. Final Formatting (Drop NaN and round)
    final_df = resampled[['open', 'high', 'low', 'close', 'volume']].dropna()
    final_df = final_df.round({'open': 1, 'high': 1, 'low': 1, 'close': 1, 'volume': 5})
    
    final_df.to_csv(output_file)
    print("Successfully generated candles. Sample:")
    print(final_df.tail())


/tmp/ipykernel_2513461/1174001660.py:15: DtypeWarning: Columns (0: 2, 1: 3, 2: 4, 3: 5, 4: 6, 5: 7, 6: 8, 7: 9, 8: 10, 9: 11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, header=None, on_bad_lines='skip')
/tmp/ipykernel_2513461/1174001660.py:15: DtypeWarning: Columns (0: 2, 1: 3, 2: 4, 3: 5, 4: 6, 5: 7, 6: 8, 7: 9, 8: 10, 9: 11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, header=None, on_bad_lines='skip')
/tmp/ipykernel_2513461/1174001660.py:15: DtypeWarning: Columns (0: 2, 1: 3, 2: 4, 3: 5, 4: 6, 5: 7, 6: 8, 7: 9, 8: 10, 9: 11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, header=None, on_bad_lines='skip')
/tmp/ipykernel_2513461/1174001660.py:15: DtypeWarning: Columns (0: 2, 1: 3, 2: 4, 3: 5, 4: 6, 5: 7, 6: 8, 7: 9, 8: 10, 9: 11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, header=Non

Successfully generated candles. Sample:
                              open     high      low    close     volume
dt                                                                      
2026-04-12 23:35:00+00:00  70542.5  70626.5  70542.5  70609.5   32.22492
2026-04-12 23:40:00+00:00  70616.5  70651.5  70582.5  70582.5   82.41380
2026-04-12 23:45:00+00:00  70697.5  70697.5  70580.0  70580.5  268.39633
2026-04-12 23:50:00+00:00  70559.5  70738.5  70559.5  70696.5   58.31790
2026-04-12 23:55:00+00:00  70690.5  70744.5  70687.5  70737.5   34.10183


In [45]:
final_df.tail()

,open,high,low,close,volume
dt,,,,,
2026-04-12 23:35:00+00:00,70542.5,70626.5,70542.5,70609.5,32.22492
2026-04-12 23:40:00+00:00,70616.5,70651.5,70582.5,70582.5,82.41380
2026-04-12 23:45:00+00:00,70697.5,70697.5,70580.0,70580.5,268.39633
2026-04-12 23:50:00+00:00,70559.5,70738.5,70559.5,70696.5,58.31790
2026-04-12 23:55:00+00:00,70690.5,70744.5,70687.5,70737.5,34.10183


In [46]:

ohlc_dict = {                                                                                                             
    'open': 'first',                                                                                                    
    'high': 'max',                                                                                                       
    'low': 'min',                                                                                                        
    'close': 'last',                                                                                                    
    'volume': 'last',
}

final_df = final_df.resample('30min', closed='left', label='left').apply(ohlc_dict)
#df = df.resample('30min', closed='left', label='left').apply(ohlc_dict)

In [26]:
df[df['coin'] == 'BTC'].head()

KeyError: 'coin'

In [47]:
final_df.tail()

,open,high,low,close,volume
dt,,,,,
2026-04-12 21:30:00+00:00,71267.5,71298.5,71178.5,71226.5,23.98130
2026-04-12 22:00:00+00:00,71331.5,71331.5,70594.5,70594.5,249.92934
2026-04-12 22:30:00+00:00,70628.5,70884.5,70544.5,70884.5,72.16720
2026-04-12 23:00:00+00:00,70883.5,70883.5,70565.5,70729.5,151.11276
2026-04-12 23:30:00+00:00,70684.5,70744.5,70542.5,70737.5,34.10183


In [ ]:
	                Open	High	Low	   Close	Volume
Open time					
2026-04-12 21:30:00	71267.0	71332.0	71171.0	71331.0	175.44249
2026-04-12 22:00:00	71332.0	71332.0	70523.0	70628.0	1768.89437
2026-04-12 22:30:00	70628.0	70904.0	70450.0	70883.0	1038.15066
2026-04-12 23:00:00	70883.0	70883.0	70555.0	70685.0	796.87852
2026-04-12 23:30:00	70684.0	70766.0	70524.0	70705.0	776.66664

In [22]:
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11
0,time,coin,funding,open_interest,prev_day_px,day_ntl_vlm,premium,oracle_px,mark_px,mid_px,impact_bid_px,impact_ask_px
1,2026-04-12T00:00:00Z,0G,-0.0000228933,897140,0.5418,74100.3254,-0.0006831461,0.55625,0.55525,0.55514,0.554378,0.55587
2,2026-04-12T00:00:00Z,2Z,0.0000125,16557008,0.08072,191472.266877,-0.0001519949,0.07895,0.078827,0.078846,0.078802,0.078938
3,2026-04-12T00:00:00Z,AAVE,0.0000125,355961,93.605,6239607.436559999,0.0005025657,94.515,94.558,94.5715,94.5625,94.589
4,2026-04-12T00:00:00Z,ACE,0.0000125,3202222.54,0.1186,47476.278182,0,0.1175,0.1174,0.1174,0.1172,0.1176


In [7]:
import requests
import pandas as pd
import time
import random
from datetime import datetime, timedelta

# Configuration
SYMBOL = 'BTCUSDC'
GRANULARITY = '5min' 
LIMIT = '200'
DAYS_TO_FETCH = 365

def fetch_all_candles(symbol, interval, days):
    url = "https://api.bitget.com/api/v2/spot/market/history-candles"
    all_candles = []
    
    # Enhanced headers to mimic a real browser more effectively
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.bitget.com/",
        "Origin": "https://www.bitget.com"
    }
    
    end_time = int(time.time() * 1000)
    start_threshold = int((datetime.now() - timedelta(days=days)).timestamp() * 1000)
    
    print(f"Starting download for {symbol}...")

    while end_time > start_threshold:
        params = {
            "symbol": symbol,
            "granularity": interval,
            "endTime": str(end_time),
            "limit": LIMIT
        }
        
        try:
            response = requests.get(url, params=params, headers=headers, timeout=20)
            
            # Check for non-200 status codes
            if response.status_code != 200:
                print(f"HTTP Error {response.status_code}. Response: {response.text[:100]}")
                break
                
            # Safe JSON parsing
            try:
                data = response.json()
            except Exception:
                print(f"Failed to parse JSON. Raw response start: {response.text[:100]}")
                break
            
            if data.get('code') == '00000' and data.get('data'):
                candles = data['data']
                all_candles.extend(candles)
                
                # Update end_time to the earliest timestamp in the batch
                earliest_ts = int(candles[-1][0]) 
                
                if earliest_ts >= end_time:
                    break
                
                end_time = earliest_ts
                print(f"Fetched until: {pd.to_datetime(earliest_ts, unit='ms')}")
                
                # Randomize sleep to appear less bot-like
                time.sleep(random.uniform(1.2, 2.5)) 
            else:
                print(f"Stopped: {data.get('msg', 'No more data')}")
                break
                
        except Exception as e:
            print(f"Connection error: {e}")
            break

    if not all_candles: return None

    cols = ['timestamp', 'open', 'high', 'low', 'close', 'base_vol', 'quote_vol', 'extra']
    df = pd.DataFrame(all_candles, columns=cols)
    df['timestamp'] = pd.to_datetime(df['timestamp'].astype(float), unit='ms')
    return df.drop_duplicates(subset=['timestamp']).sort_values('timestamp')

# Execution
df_year = fetch_all_candles(SYMBOL, GRANULARITY, DAYS_TO_FETCH)
if df_year is not None:
    df_year.to_csv("bitget_btcusdc_full_year.csv", index=False)
    print(f"\nSuccess! Total Candles: {len(df_year)}")


Starting download for BTCUSDC...
Fetched until: 2026-04-26 06:00:00
Fetched until: 2026-04-26 05:55:00
Fetched until: 2026-04-26 05:50:00
Fetched until: 2026-04-26 05:45:00
Fetched until: 2026-04-26 05:40:00
Fetched until: 2026-04-26 05:35:00
Fetched until: 2026-04-26 05:30:00
Fetched until: 2026-04-26 05:25:00
Fetched until: 2026-04-26 05:20:00
Fetched until: 2026-04-26 05:15:00
Fetched until: 2026-04-26 05:10:00
Fetched until: 2026-04-26 05:05:00
Fetched until: 2026-04-26 05:00:00
Fetched until: 2026-04-26 04:55:00
Fetched until: 2026-04-26 04:50:00
Fetched until: 2026-04-26 04:45:00
Fetched until: 2026-04-26 04:40:00
Fetched until: 2026-04-26 04:35:00
Fetched until: 2026-04-26 04:30:00
Fetched until: 2026-04-26 04:25:00
Fetched until: 2026-04-26 04:20:00
Fetched until: 2026-04-26 04:15:00
Fetched until: 2026-04-26 04:10:00
Fetched until: 2026-04-26 04:05:00
Fetched until: 2026-04-26 04:00:00
Fetched until: 2026-04-26 03:55:00
Fetched until: 2026-04-26 03:50:00
Fetched until: 2026-04

KeyboardInterrupt: 

In [1]:
import requests
API_KEY = "cede3b36-6097-42a6-9e1e-5e55dd0ada53"
BASE_URL = f"https://dwellir.com{API_KEY}"

print(f"DEBUG: I am trying to connect to: {BASE_URL}")

# Check if it looks right. It MUST have a slash after .com and after /ohlcv/
# If it looks right, let's try a single test request
payload = {"method": "get_ohlcv", "params": {"symbol": "UBTC", "interval": "5m", "limit": 1}}
r = requests.post(BASE_URL, json=payload)
print(f"STATUS: {r.status_code}")
print(f"DATA: {r.json()}")

DEBUG: I am trying to connect to: https://dwellir.comcede3b36-6097-42a6-9e1e-5e55dd0ada53


ConnectionError: HTTPSConnectionPool(host='dwellir.comcede3b36-6097-42a6-9e1e-5e55dd0ada53', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='dwellir.comcede3b36-6097-42a6-9e1e-5e55dd0ada53', port=443): Failed to resolve 'dwellir.comcede3b36-6097-42a6-9e1e-5e55dd0ada53' ([Errno -2] Name or service not known)"))